# Update log
2024/08/29
- Test without Responsivity and/or baseline
- Test with reduced number of features
- Baseline alone gives up to 75% accuracy
- Suspect data distribution due to always running experiment in 0,1,2,3,4
- Run experiment in De Bruijn sequence to balance the adjacent experiment channels
---

In [63]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import json

spk_data = "D:\\code\\uom_explore\\model_input\\3_features.csv"
spk_pca_data = "D:\\code\\uom_explore\\model_input\\pca_df.csv"
spk_20 = "D:\\code\\uom_explore\\data_science\\reduced\\20_140_195_df.csv"
spk_full = "D:\\code\\uom_explore\\data_science\\df.csv"
debruijn_1 = "D:\\code\\uom_explore\\processed_data\\metrics_exp_brujin_seq_1.csv"

hkr_wsl_data = "/home/hk-wsl/code/uom_explore/model_input/feature_matrix.csv"
hkr_pca_data = "/home/hk-wsl/code/uom_explore/model_input/feature_pca.csv"

spk_json = "/home/gavinlouuu/coding/uom_explore/data_science/parameter.json"
hkr_wsl_json = "/home/hk-wsl/code/uom_explore/data_science/parameter.json"

data_path = debruijn_1
param_path = spk_json

with open('parameter.json','r') as file:
    params = json.load(file)

# Hyperparameters
# Extract parameters from the JSON object
hidden_size = params['hidden_size']
ground_truth = params['ground_truth']
num_epochs = params['num_epochs']
batch_size = params['batch_size']
learning_rate = params['learning_rate']
momentum_value = params['momentum_value']
dropout_rate = params['dropout']

df = pd.read_csv(data_path)
# drop experiment_id column
# df.drop('experiment_id', axis=1, inplace=True)
print(type(df))
df.head()



<class 'pandas.core.frame.DataFrame'>


,experiment_id,channel_id,baseline_160,baseline_162,baseline_165,baseline_167,baseline_170,baseline_172,baseline_175,baseline_177,...,temperature_max,temperature_std,humidity_mean,humidity_min,humidity_max,humidity_std,pressure_mean,pressure_min,pressure_max,pressure_std
0,20240829145200s1c0r0,0,15333.197116,22957.351691,23848.697631,23071.544934,21668.667869,20477.232045,19239.758925,18185.434345,...,33.21,0.333116,64.992000,60.89,68.48,2.459250,100210.552941,100201.0,100213.0,1.802814
1,20240829145235s1c1r0,1,21815.038599,33226.561073,34565.623032,33401.419655,31172.487756,29492.245969,27552.348528,25827.395031,...,33.29,0.036399,62.682083,60.15,65.88,1.926198,100210.319444,100209.0,100212.0,0.667693
2,20240829145311s1c2r0,2,28245.293614,44118.401113,45736.157153,43995.030951,41154.533173,38596.182230,35870.663615,33576.358547,...,33.40,0.039994,59.122069,58.56,59.73,0.344570,100210.344828,100208.0,100212.0,0.899969
3,20240829145346s1c3r0,3,32950.229651,52038.602759,54191.269085,51752.884670,47946.761953,45108.761385,42011.398789,39147.586067,...,33.44,0.029688,59.461351,58.47,60.65,0.728982,100207.351351,100204.0,100209.0,1.127884
4,20240829145422s1c4r0,4,18762.637612,28167.472573,28372.246360,26803.690386,24567.917448,22757.383722,20645.451620,19187.641379,...,33.49,0.029308,60.926180,58.38,64.68,2.028016,100207.573034,100205.0,100210.0,1.185975


## Load all features

In [64]:
# Get all column names from the DataFrame
all_columns = df.columns.tolist()

# Remove 'channel_id' and the ground truth from the list of features
features = [col for col in all_columns if col != 'experiment_id' and col != ground_truth]

# Print the features
print("Features:")
print(json.dumps(features, indent=2))



Features:
[
  "baseline_160",
  "baseline_162",
  "baseline_165",
  "baseline_167",
  "baseline_170",
  "baseline_172",
  "baseline_175",
  "baseline_177",
  "baseline_180",
  "baseline_182",
  "baseline_185",
  "baseline_187",
  "baseline_190",
  "baseline_192",
  "baseline_195",
  "baseline_197",
  "baseline_200",
  "baseline_202",
  "baseline_205",
  "baseline_210",
  "baseline_212",
  "baseline_215",
  "baseline_217",
  "baseline_220",
  "baseline_222",
  "baseline_225",
  "baseline_227",
  "baseline_230",
  "baseline_232",
  "baseline_235",
  "baseline_237",
  "max_reaction_R_160",
  "max_reaction_R_162",
  "max_reaction_R_165",
  "max_reaction_R_167",
  "max_reaction_R_170",
  "max_reaction_R_172",
  "max_reaction_R_175",
  "max_reaction_R_177",
  "max_reaction_R_180",
  "max_reaction_R_182",
  "max_reaction_R_185",
  "max_reaction_R_187",
  "max_reaction_R_190",
  "max_reaction_R_192",
  "max_reaction_R_195",
  "max_reaction_R_197",
  "max_reaction_R_200",
  "max_reaction_R_202"

# Keep all features

In [65]:
# X includes all features
X = df[features]



# PCA processing

In [66]:
from sklearn.decomposition import PCA

# Determine the number of components to keep 95% of the variance
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X)

# Update the input size for the model
input_size = X_pca.shape[1]

print(f"Number of PCA components: {input_size}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

# Update X to use PCA-transformed data
X = pd.DataFrame(X_pca, columns=[f'PC_{i+1}' for i in range(input_size)])


Number of PCA components: 10
Explained variance ratio: [9.97016215e-01 2.63442857e-03 2.54491244e-04 4.60200582e-05
 7.71330301e-06 6.51392103e-06 4.89050196e-06 4.28765880e-06
 3.16297061e-06 2.66773461e-06]


# Data split and scale

In [67]:
# Preview features
# X = X[features]
print(X.head())


            PC_1          PC_2         PC_3         PC_4         PC_5  \
0 -476191.208506 -21828.113488  9297.990123 -2125.588694   229.262917   
1 -429756.469181 -26657.949179  6449.963761  -520.477873   408.819777   
2 -383527.043530 -30154.216306  3220.417223   882.875646   677.315016   
3 -349887.618093 -32158.711249  1432.530311  1355.346805   800.011850   
4 -475810.188696  -6859.733402  7087.137853 -4161.556598 -1132.206063   

         PC_6         PC_7        PC_8        PC_9       PC_10  
0 -400.793114  -449.401965  218.884814   25.424975  730.642096  
1 -359.026470   -45.486006  121.997847  -14.230264  436.314054  
2 -239.254704   -40.172983 -136.795664  -83.389766  416.991158  
3 -275.979801   164.831031  112.145255   84.228189  154.773801  
4 -585.168780 -1219.557055  931.140964 -555.701926  282.201302  


In [68]:
input_size = len(X.columns)  # removing the ground truth from the number of columns counted
num_classes = df[ground_truth].nunique()
print(f"Number of classes: {num_classes}")
print(f"Number of features: {input_size}")

# Preview ground truth
y = df[ground_truth]

# Split into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)  # This makes 60%, 20%, 20%

# Initialize the StandardScaler
scaler = StandardScaler()
# scaler = MinMaxScaler(feature_range=(0,255)) # 

# Fit the scaler to the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert arrays to tensors
X_train_scaled = torch.tensor(X_train_scaled, dtype=torch.float32).unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long)  # Convert to NumPy array first
X_val_scaled = torch.tensor(X_val_scaled, dtype=torch.float32).unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_val = torch.tensor(y_val.to_numpy(), dtype=torch.long)  # Convert to NumPy array first
X_test_scaled = torch.tensor(X_test_scaled, dtype=torch.float32).unsqueeze(1)  # Shape: [batch_size, 1, num_features]
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.long)  # Convert to NumPy array first

# Create datasets
train_dataset = TensorDataset(X_train_scaled, y_train)
val_dataset = TensorDataset(X_val_scaled, y_val)
test_dataset = TensorDataset(X_test_scaled, y_test)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Number of classes: 5
Number of features: 10


# 1DCNN

In [69]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

in_channels = params['in_channels']
out_channels = params['out_channels']
kernel_sizes = params['kernel_sizes']

class CNN1DClassifier(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes, num_classes, dropout_rate=0.5):
        super(CNN1DClassifier, self).__init__()
        
        assert len(out_channels) == len(kernel_sizes), "The length of out_channels and kernel_sizes must be the same"
        
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()  # Adding batch normalization if needed
        
        current_in_channels = in_channels
        
        for out_channel, kernel_size in zip(out_channels, kernel_sizes):
            self.convs.append(nn.Conv1d(current_in_channels, out_channel, kernel_size=kernel_size, stride=1, padding=kernel_size // 2))
            self.bns.append(nn.BatchNorm1d(out_channel))  # Optional: Add batch normalization
            current_in_channels = out_channel
        
        # Calculate the size after all convolutional and pooling layers
        conv_output_size = input_size
        for kernel_size in kernel_sizes:
            conv_output_size = (conv_output_size + 2 * (kernel_size // 2) - (kernel_size - 1) - 1) // 1 + 1
            conv_output_size = conv_output_size // 2  # After pooling
        
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(out_channels[-1] * conv_output_size, 128)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(128, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # print(f'Input shape: {x.shape}')
        for conv, bn in zip(self.convs, self.bns):
            x = self.pool(torch.relu(bn(conv(x))))
            # print(f'After conv and pool: {x.shape}')
        x = x.view(x.size(0), -1)  # Flatten the tensor
        # print(f'After flatten: {x.shape}')
        x = torch.relu(self.fc1(x))
        # print(f'After fc1: {x.shape}')
        x = self.dropout(x)
        x = self.fc2(x)
        # print(f'After fc2: {x.shape}')
        x = self.softmax(x)
        return x

# Train the 1D CNN model

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=25, device='cpu'):
    model = model.to(device)
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)

        print(f'Epoch {epoch}/{num_epochs - 1}')
        print(f'Training Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_corrects.double() / len(val_loader.dataset)

        print(f'Validation Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    return model

# Initialize model, loss function, and optimizer
model = CNN1DClassifier(in_channels, out_channels, kernel_sizes, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
trained_model = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=num_epochs)

# Evaluate the model on the test set
model.eval()
test_loss = 0.0
test_corrects = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * inputs.size(0)
        test_corrects += torch.sum(preds == labels.data)

test_loss = test_loss / len(test_loader.dataset)
test_acc = test_corrects.double() / len(test_loader.dataset)

print(f'Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}')


Epoch 0/199
Training Loss: 1.5940 Acc: 0.2750
Validation Loss: 1.6053 Acc: 0.2250
Epoch 1/199
Training Loss: 1.4885 Acc: 0.5250
Validation Loss: 1.5889 Acc: 0.4250
Epoch 2/199
Training Loss: 1.3838 Acc: 0.6250
Validation Loss: 1.5432 Acc: 0.4250
Epoch 3/199
Training Loss: 1.3293 Acc: 0.6833
Validation Loss: 1.4795 Acc: 0.5000
Epoch 4/199
Training Loss: 1.2654 Acc: 0.7333
Validation Loss: 1.4410 Acc: 0.4500
Epoch 5/199
Training Loss: 1.2408 Acc: 0.7333
Validation Loss: 1.4218 Acc: 0.5000
Epoch 6/199
Training Loss: 1.1846 Acc: 0.7917
Validation Loss: 1.4050 Acc: 0.5000
Epoch 7/199
Training Loss: 1.1521 Acc: 0.8167
Validation Loss: 1.3885 Acc: 0.5250
Epoch 8/199
Training Loss: 1.1206 Acc: 0.8167
Validation Loss: 1.3611 Acc: 0.5750
Epoch 9/199
Training Loss: 1.0765 Acc: 0.8583
Validation Loss: 1.3367 Acc: 0.6250
Epoch 10/199
Training Loss: 1.0578 Acc: 0.8750
Validation Loss: 1.3352 Acc: 0.6000
Epoch 11/199
Training Loss: 1.0323 Acc: 0.9333
Validation Loss: 1.3346 Acc: 0.6000
Epoch 12/199
T